# mmm_ready × Meridian: full run on Colab

Builds `mmm_input_weekly` from the synthetic data with dbt, fits Meridian with the `--full`
sampling preset, and validates the fit against the known ground truth (ROI per channel,
marginal ROI and the budget optimum). See `examples/meridian/README.md` for the method and
the convergence criterion.

- Runtime: **Runtime → Change runtime type → GPU** is recommended (a T4 is enough), but CPU also works.
- Nothing here needs credentials: the repo is public and the data is generated in the runtime.
- This is one run (seed 42) on synthetic data. The thresholds are heuristics.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/ceterisdecisions/mmm_ready.git"
BRANCH = "main"  # change to test another branch
REPO_DIR = "/content/mmm_ready"

In [ ]:
!nvidia-smi || echo 'No GPU: running on CPU'

## 2. Clone and build the data with dbt

In [ ]:
!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline
!pip install -q -r requirements-dev.txt

In [ ]:
# Default package vars only: Monday weeks, only channels with spend, partial edge weeks dropped.
%cd {REPO_DIR}/integration_tests
%env DBT_PROFILES_DIR=.
!python ../scripts/generate_synthetic_data.py --out seeds
!dbt deps && dbt seed && dbt build
%cd {REPO_DIR}

## 3. Install Meridian

If pip reports conflicts with Colab's preinstalled packages, use **Runtime → Restart session**
and continue from the next cell (it changes back into the repo).

In [ ]:
!pip install -q "google-meridian[and-cuda,schema]==2.2.0" duckdb matplotlib

In [ ]:
REPO_DIR = "/content/mmm_ready"
%cd {REPO_DIR}
import jax
print(jax.devices())

## 4. Quick run (timing reference)

The quick preset (2 chains × 600 steps) finishes in about a minute on a laptop CPU. Its sampling
time gives a rough estimate for the full run (4 chains × 2000 steps). It is written to a separate
folder and is not used for validation.

In [ ]:
!python examples/meridian/run_meridian.py --out examples/meridian/output_quick

## 5. Full run

Same model, priors, knots and seed; only the sampling preset changes
(4 chains, 500 adapt, 500 burn-in, 1000 kept draws, 500 prior draws).
The convergence verdict is printed at the end and saved to `output/run_config.json`.

In [ ]:
!python examples/meridian/run_meridian.py --full

In [ ]:
!python examples/meridian/response_curves.py
!python examples/meridian/optimize_budget.py

## 6. Validation against the ground truth

In [ ]:
!python examples/meridian/validate.py > /dev/null
from IPython.display import Markdown, display
display(Markdown(open("examples/meridian/output/validation_report.md").read()))

In [ ]:
from IPython.display import Image
for name in ["response_curves", "saturation_normalized", "mroi", "adstock", "hill_curves", "budget_allocation"]:
    display(Image(f"examples/meridian/output/plots/{name}.png"))

## 7. Download the outputs

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/meridian_output", "zip", "examples/meridian/output")
files.download(archive)